# Appendix A: Jupyter Notebook 1 (Base) (No Parallelisation)

In [ ]:
from medmnist import BloodMNIST
import torch
from torch import nn, utils, Generator
import torch.nn.functional as F
import torchvision
from tqdm.notebook import trange, tqdm
import plotly.express as px
from typing import Sequence
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"
import plotly.colors as pc

In [ ]:
import plotly.io as pio
pio.get_chrome()

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
EPOCHS = 30

## Dataset Preparation

[MedMNIST](https://doi.org/10.1038/s41597-022-01721-8) is a collection of 18x standardized datasets for 2D and 3D biomedical image classification. 

It contains multiple size options: 28 (MNIST-Like), 64, 128, and 224.

To download:
```bash
sudo apt-get install aria2
aria2c -x 16 -s 16 "https://zenodo.org/records/10519652/files/pathmnist_128.npz?download=1" -o /var/tmp/pathmnist_128.npz
```

In [ ]:
prepare_dataset = lambda x: BloodMNIST(x, torchvision.transforms.ToTensor(), lambda x:x.squeeze(), download=True, root="/var/tmp", size=128)
ds_train, ds_val, ds_test = [prepare_dataset(x) for x in ['train', 'val', 'test']]

random_generator = Generator().manual_seed(42)
prepare_dataloader = lambda x: utils.data.DataLoader(x, 64, True, generator=random_generator)

dl_train, dl_val, dl_test = [prepare_dataloader(x) for x in [ds_train, ds_val, ds_test]]

classes, label_lookup = len(ds_train.info['label']), ds_train.info['label']
label_lookup[str(3)] = "immature granulocytes" # I wanted a shorter name

ds_train

In [ ]:
ex = next(iter(dl_train))
print(ex[0].shape, ex[1].shape)

img = torch.einsum("nchw->nhwc",ex[0][:10])
fig = px.imshow(img, binary_string=True, facet_col=0, facet_col_wrap=5, facet_col_spacing=0.01, facet_row_spacing=0.08)

item_map={f'{i}':label_lookup[str(key.item())] for i, key in enumerate(ex[1][:10])}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]])) 

fig.update_layout(
        # title={"text": "N", "x": 0.5},
        height=600,
        width=1500,
        margin=dict(l=20, r=20, t=20, b=20),
    )

fig.write_image(f"figures/sample_images_blood_mnist.svg")

fig.show()

## Model Definitions

In [ ]:
class SoftmaxRegression(nn.Module):
    def __init__(self, outfeatures: int, bias = True):
        super().__init__()
        self.f = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(outfeatures, bias)
        )

    def forward(self, x):
        return self.f(x)

class CNNBlock(nn.Module):
    def __init__(self, out_channels:int, conv_layers=4, skip_frequency=2, norm=True):
        super().__init__()
        self.conv_layers = conv_layers
        self.features = nn.ModuleList()
        self.residuals = nn.ModuleList()
        self.skip_frequency = skip_frequency
        
        for i in range(conv_layers):
            self.features.append(nn.Sequential(nn.LazyConv2d(out_channels, 3, padding=1), *([nn.GroupNorm(1, out_channels)] if norm else []), nn.ReLU()))
            if self._should_end_skip(i):
                self.residuals.append(nn.LazyConv2d(out_channels, 1))

    def _has_skips(self):
        return self.skip_frequency != 0

    def _should_start_skip(self, i):
        return self._has_skips() and (i%self.skip_frequency == 0 and (i+self.skip_frequency) <= self.conv_layers)
    
    def _should_end_skip(self, i):
        return self._has_skips() and ((i+1)%self.skip_frequency == 0)

    def forward(self, x):
        identity = 0
        for i, conv in enumerate(self.features):
            if self._should_start_skip(i):
                identity = self.residuals[i//self.skip_frequency](x)
            x = conv(x)
            if self._should_end_skip(i):
                x = x + identity
        return x

class CNN(nn.Module):
    r"""Construct VGG/ResNet hybrid CNN

    Core architecture of VGG remains the same, with the addition of skip
    connections within each block.

    Standard features of ResNet such as strides conv instead of max pool,
    global average pooling instead of FC layers and bottleneck blocks have
    not been included.
    
    Args:
        arch: List of tuples. Each tuple contains number of convolutions in layer, out channels of layer,
            residual frequency in layer. Set residual frequency to 0 to disable residuals.
        classes: Number of logits.
        dropout_rate: Dropout used in the classifier layer.
        norm: Whether to enable group normalisation.
    """

    def __init__(self, arch: list[tuple[int,int,int]], classes=10, dropout_rate=0.0, norm=True):
        super().__init__()

        self.max_pool = nn.MaxPool2d(2)
        self.dropout_rate = dropout_rate
        self.features = nn.ModuleList()
        self.arch = arch

        layers = []
        for (num_convs, out_ch, residuals) in self.arch:
            layers.append(CNNBlock(out_ch, num_convs, residuals, norm))
            layers.append(self.max_pool)

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.LazyLinear(classes),
        )

        # ResNet style classifier.
        # self.classifier = nn.Sequential(
        #     nn.AdaptiveAvgPool2d(1),
        #     nn.Flatten(),
        #     nn.LazyLinear(classes),
        # )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
class Job():
    def __init__(self, model: nn.Module, name: str, epochs: int, lr: float, batch_size: int):
        self.model = model
        self.name = name
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size

class Record():
    def __init__(self):
        self.store = {}
        self.epoch_store = {}

    def add(self, name, loss, logits, targets):
        self.store.setdefault(f"{name}-loss", []).append(loss)
        acc = (logits.argmax(1) == targets).float().mean().item()
        self.store.setdefault(f"{name}-acc", []).append(acc)

    def compute(self):
        for key, values in self.store.items():
            self.epoch_store.setdefault(key, []).append(
                sum(values) / len(values)
            )
        self.store = {}

    def max_acc(self, name="train"):
        return max(self.epoch_store.get(f"{name}-acc", []))

    def max_loss(self, name="train"):
        return max(self.epoch_store.get(f"{name}-loss", []))

    def clear(self):
        self.store = {}
        self.epoch_store = {}

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re

# Generates training plot curves
def plot_comparison(metrics: Record, title="Model Comparison", splits=("val", "train"), measures=("acc", "loss"), split_subplots=False, file_name=""):
    titles = {
        "acc": "Accuracy",
        "loss": "Loss",
        "val": "Validation",
        "train": "Train"
    }

    if split_subplots:
        rows = len(splits)
        cols = len(measures)
        subplot_titles = [f"{titles[s]} {titles[m]} vs Epoch" for s in splits for m in measures]
    else:
        rows = 1
        cols = len(measures)
        subplot_titles = [f"{titles[m]} vs Epoch" for m in measures]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles, horizontal_spacing=0.04, vertical_spacing=0.08)
    models = set(k.rsplit("-", 2)[0] for k in metrics.epoch_store.keys())
    colors = pc.qualitative.Plotly

    def natural_sort_key(s):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

    trace_idx = 0
    shown = set()
    for name in sorted(models, key=natural_sort_key):
        if split_subplots:
            color = colors[trace_idx % len(colors)]
            trace_idx += 1
            for split in splits:
                for m_idx, measure in enumerate(measures):
                    key = f"{name}-{split}-{measure}"
                    values = metrics.epoch_store.get(key)
                    if values is None:
                        continue
                    epochs = list(range(len(values)))
                    show = name not in shown
                    shown.add(name)
                    fig.add_trace(
                        go.Scatter(x=epochs, y=values, name=name, legendgroup=name, showlegend=show, line=dict(color=color)),
                        row=splits.index(split) + 1, col=m_idx + 1
                    )
        else:
            for split in splits:
                color = colors[trace_idx % len(colors)]
                trace_idx += 1
                label = f"{name} ({split})" if len(splits) > 1 else name
                for m_idx, measure in enumerate(measures):
                    key = f"{name}-{split}-{measure}"
                    values = metrics.epoch_store.get(key)
                    if values is None:
                        continue
                    epochs = list(range(len(values)))
                    show = label not in shown
                    shown.add(label)
                    fig.add_trace(
                        go.Scatter(x=epochs, y=values, name=label, legendgroup=label, showlegend=show, line=dict(color=color)),
                        row=1, col=m_idx + 1
                    )

    fig.update_xaxes(title_text="Epoch")
    for m_idx, measure in enumerate(measures):
        for r in range(1, rows + 1):
            fig.update_yaxes(title_text=titles[measure], row=r, col=m_idx + 1)

    fig.update_layout(
        title={"text": title, "x": 0.5},
        showlegend=len(models) > 1 or len(splits) > 1,
        height=600 * rows,
        width=1500
    )
    if file_name:
        fig.write_image(f"figures/{file_name}.svg")
    fig.show()

In [ ]:
def train_model(job: Job, metric_store: Record, device: torch.device):
    job.model.to(device)
    dl = utils.data.DataLoader(ds_train, job.batch_size, True, generator=Generator().manual_seed(42))
    trainer = torch.optim.Adam(job.model.parameters(), lr=job.lr)
    loss = nn.CrossEntropyLoss()

    with trange(job.epochs) as t:
        t.set_description(job.name)
        for i in t:
            job.model.train()
            for X, y in tqdm(dl, leave=False, desc="train"):
                X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = job.model(X)
                l = loss(out, y)
                trainer.zero_grad()
                l.backward()
                trainer.step()
                metric_store.add(f"{job.name}-train", l.item(), out.detach(), y)

            job.model.eval()
            with torch.no_grad():
                for X, y in tqdm(dl_val, leave=False, desc="val"):
                    X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
                    
                    out = job.model(X)
                    l = loss(out, y)
                    metric_store.add(f"{job.name}-val", l.item(), out.detach(), y)

            metric_store.compute()
            t.set_description(f"{job.name} ({max(metric_store.epoch_store[f"{job.name}-val-acc"])*100:.0f}%)")

    job.model.cpu()
    torch.cuda.empty_cache()

def run_jobs(jobs: Sequence[Job], device: torch.device):
    metric_store = Record()
    for job in jobs:
        train_model(job, metric_store, device)
        test_loss, test_acc = test_model(job.model, device)
        val_acc = metric_store.max_acc(f"{job.name}-val")
        train_acc = metric_store.max_acc(f"{job.name}-train")
        print(f"{job.name}: test acc={test_acc} val acc={val_acc} train acc={train_acc}")

    return metric_store

def test_model(model: nn.Module, device: torch.device):
    metrics = Record()
    model.to(device)
    model.eval()
    loss = nn.CrossEntropyLoss()
    with torch.no_grad():
        for X, y in tqdm(dl_test, leave=False, desc="test"):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            out = model(X)
            l = loss(out, y)
            metrics.add(f"", l.item(), out.detach(), y)
    metrics.compute()
    return (metrics.epoch_store["-loss"][0], metrics.epoch_store["-acc"][0])

# def test_models(jobs: Sequence[Job], device: torch.device):
#     out = "Test summary metrics:\n"
#     for job in jobs:
#         test_loss, test_acc = test_model(job.model, device)
#         out += f"  {job.name}: loss={test_loss:.2f}, acc={test_acc*100:.2f}%\n"
#     print(out)

In [ ]:
device = torch.device('cuda')

## Softmax

In [ ]:
linear = SoftmaxRegression(classes)
softmax_job = Job(
    linear,
    "softmax",
    EPOCHS,
    1e-3,
    64
)
softmax_metrics = run_jobs([softmax_job], device)

In [ ]:
plot_comparison(softmax_metrics, "Softmax performance on BloodMNIST (128x128)", file_name="softmax_bloodmnist")

## Convolutional Neural Network

### Batchnorm off

In [ ]:
model2_norm_off_job = Job(
    CNN(arch=[(1,32,0), (1,64,0)], classes=classes, norm=False), # 1+1
    "cnn-2",
    2,
    1e-3,
    64
)

model8_norm_off_job = Job(
    CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, norm=False),
    "cnn-8",
    2,
    1e-3,
    64
)

model16_norm_off_job = Job(
    CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, norm=False),
    "cnn-16",
    EPOCHS,
    1e-3,
    64
)

model32_norm_off_job = Job(
    CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, norm=False),
    "cnn-32",
    EPOCHS,
    1e-3,
    64
)

jobs_batchnorm_off = [model2_norm_off_job, model8_norm_off_job]
metrics_batchnorm_off = run_jobs(jobs_batchnorm_off, device)

In [ ]:
plot_comparison(metrics_batchnorm_off, "CNN Comparison Normalisation Off", splits=("val","train"), split_subplots=True)

### Batchnorm on

In [ ]:
model2_norm_on_job = Job(
    CNN(arch=[(1,32,0), (1,64,0)], classes=classes, norm=True),
    "cnn-2",
    EPOCHS,
    1e-3,
    64
)

model8_norm_on_job = Job(
    CNN(arch=[(3,32,0), (3,64,0), (2,128,0)], classes=classes, norm=True),
    "cnn-8",
    EPOCHS,
    1e-3,
    64
)

model16_norm_on_job = Job(
    CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, norm=True),
    "cnn-16",
    EPOCHS,
    1e-3,
    64
)

model32_norm_on_job = Job(
    CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, norm=True),
    "cnn-32",
    EPOCHS,
    1e-3,
    64
)

jobs_batchnorm_on = [model2_norm_on_job, model8_norm_on_job, model16_norm_on_job, model32_norm_on_job]
metrics_batchnorm_on = run_jobs(jobs_batchnorm_on, device)

In [ ]:
# import copy
# filtered_metrics = copy.deepcopy(metrics_batchnorm_on)
# for key in filtered_metrics.epoch_store:
#     filtered_metrics.epoch_store[key] = filtered_metrics.epoch_store[key][0:30]

In [ ]:
plot_comparison(metrics_batchnorm_on, "CNN Performance Comparison on BloodMNIST (128x128)", splits=("val","train"), split_subplots=True, file_name="cnn_comparison_bloodmnist")

### Improving Deeper Model

In [ ]:
model16_base_job = Job(
    CNN(arch=[(6,32,0), (5,64,0), (5,128,0)], classes=classes, norm=True),
    "cnn-16",
    EPOCHS,
    1e-3,
    64
)

model32_base_job = Job(
    CNN(arch=[(8,32,0), (8,64,0), (8,128,0), (8,256,0)], classes=classes, norm=True),
    "cnn-32",
    EPOCHS,
    1e-3,
    64
)

model16_improved_job = Job(
    CNN(arch=[(6,32,2), (5,64,2), (5,128,2)], classes=classes, dropout_rate=0.5, norm=True),
    "cnn-16-improved",
    EPOCHS,
    1e-3,
    64
)

model32_improved_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "cnn-32-improved",
    EPOCHS,
    1e-3,
    64
)

jobs_improvement_comparison = [model16_base_job, model32_base_job, model16_improved_job, model32_improved_job]
metrics_improvement_comparison = run_jobs(jobs_improvement_comparison, device)

In [ ]:
model16_improved_job = Job(
    CNN(arch=[(6,32,2), (5,64,2), (5,128,2)], classes=classes, dropout_rate=0.5, norm=True),
    "cnn-16-improved",
    EPOCHS,
    1e-3,
    64
)


model32_improved_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "cnn-32-improved",
    EPOCHS,
    1e-3,
    64
)


from torchinfo import summary

print(summary(model16_improved_job.model, input_size=(64, 3, 128, 128)))
print(summary(model32_improved_job.model, input_size=(64, 3, 128, 128)))

In [ ]:
plot_comparison(metrics_improvement_comparison, "CNN-I Performance Comparison on BloodMNIST (128x128)", splits=("val","train"), split_subplots=True, file_name="cnn_comparison_improved_bloodmnist")

### Learning Rate Comparison

In [ ]:
lr_1e6_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "lr-1e-6",
    EPOCHS,
    1e-6,
    64
)

lr_1e4_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "lr-1e-4",
    EPOCHS,
    1e-4,
    64
)

lr_1e3_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "lr-1e-3",
    EPOCHS,
    1e-3,
    64
)

lr_1e2_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "lr-1e-2",
    EPOCHS,
    1e-2,
    64
)

lr_1e0_job = Job(
    CNN(arch=[(8,32,2), (8,64,2), (8,128,2), (8,256,2)], classes=classes, dropout_rate=0.5, norm=True),
    "lr-1e0",
    EPOCHS,
    1.0,
    64
)

jobs_lr_comparison = [lr_1e6_job, lr_1e4_job, lr_1e3_job, lr_1e2_job, lr_1e0_job]
metrics_lr_comparison = run_jobs(jobs_lr_comparison, device)

In [ ]:
plot_comparison(metrics_lr_comparison, "CNN Learning Rate Comparison", splits=("val","train"), split_subplots=True, file_name="cnn_learning_rate_2_blood_mnist")

### Minibatch Size Comparison

In [ ]:
batch_size_learning_rate = 5e-4
batch_size_epochs = 10
batch_size_arch = [(8,32,2), (8,64,2), (8,128,2), (8,256,2)]

bs_1_job = Job(
    CNN(arch=batch_size_arch, classes=classes, dropout_rate=0.5, norm=True),
    "bs-1",
    batch_size_epochs,
    batch_size_learning_rate,
    1
)

bs_8_job = Job(
    CNN(arch=batch_size_arch, classes=classes, dropout_rate=0.5, norm=True),
    "bs-8",
    batch_size_epochs,
    batch_size_learning_rate,
    8
)

bs_16_job = Job(
    CNN(arch=batch_size_arch, classes=classes, dropout_rate=0.5, norm=True),
    "bs-16",
    batch_size_epochs,
    batch_size_learning_rate,
    16
)

bs_64_job = Job(
    CNN(arch=batch_size_arch, classes=classes, dropout_rate=0.5, norm=True),
    "bs-64",
    batch_size_epochs,
    batch_size_learning_rate,
    64
)

bs_256_job = Job(
    CNN(arch=batch_size_arch, classes=classes, dropout_rate=0.5, norm=True),
    "bs-256",
    batch_size_epochs,
    batch_size_learning_rate,
    256
)

jobs_bs_comparison = [bs_1_job, bs_8_job, bs_16_job, bs_64_job, bs_256_job]
metrics_bs_comparison = run_jobs(jobs_bs_comparison, device)

In [ ]:
plot_comparison(metrics_bs_comparison, "CNN Minibatch Size Comparison", splits=("val","train"), split_subplots=True, file_name="cnn_minibatch_blood_mnist")

In [ ]:
# Used to visualise classifications

# X, y = next(iter(dl_test))
# m = nn.Softmax(dim=1)
# # If model on gpu
# # X_gpu = X.to(device, non_blocking=True)
# # y_hat_probs = m(model(X_gpu)).cpu()

# y_hat_probs = m(model32improved(X))

# y_hat = y_hat_probs.argmax(axis=1)

# img = torch.einsum("nchw->nhwc",X)
# fig = px.imshow(img, binary_string=True, facet_col=0, facet_col_wrap=8, height=2000, facet_col_spacing=0.01, facet_row_spacing=0.01)

# for i, label in enumerate(y_hat):
#     text = f"{label_lookup[str(label.item())]} ({y_hat_probs[i][label]*100:.2f}%)"
#     if label != y[i]:
#         text = f"<span style='color:red'>{text}</span>"
    
#     item_map[f'{i}'] = text

# fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]])) 

# fig.show()